# Silver Layer — Employees
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.employees`, applies cleaning and null filling,
and writes the curated result to `salesflow_dev.silver.employees`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `EmployeeID` |
| 2 | Clean `FirstName`, `LastName`, `City`, `Country` |
| 3 | Fill nulls in optional fields |
| 4 | Add `data_quality_status` flag |
| 5 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.table("salesflow_dev.bronze.employees")
print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates

In [0]:
df = df.dropDuplicates(["EmployeeID"])
print(f"Records after deduplication: {df.count()}")

## 3. Clean and Standardize Columns

In [0]:
# Clean name and location columns
df = clean_string_column(df, "FirstName")
df = clean_string_column(df, "LastName")
df = clean_string_column(df, "City")


## 4. Add Quality Flag
`INVALID` if `EmployeeID`, `FirstName`, or `LastName` is null.

In [0]:
df = add_quality_flag(df, ["EmployeeID", "FirstName", "LastName"])

## 5. Add Processing Timestamp

In [0]:
df = df.withColumn("processing_timestamp", current_timestamp())

## 6. Save as Delta Table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("salesflow_dev.silver.employees")
print("Table saved: salesflow_dev.silver.employees")

## 7. Validation

In [0]:
silver_employees = spark.table("salesflow_dev.silver.employees")
print(f"Total records: {silver_employees.count()}")
print("\nQuality flag distribution:")
display(silver_employees.groupBy("data_quality_status").count())
print("\nSchema:")
silver_employees.printSchema()
print("\nFirst 5 rows:")
display(silver_employees.limit(5))